1. django-admin startapp pracdjango

2. pracdjango/models.py 写关于数据库的内容

3. settings.py - INSTALLED_APPS 加入 pradjango

4. python3 manage.py makemigrations

5. python3 manage.py migrate



models.py

from django.db import models

# Create your models here.

class IPRequestLog(models.Model): # 不要忘记加 models.Model!!!
    """This is to record ip address apply log"""

    # id = models.AutoField(primary_key=True) Django 默认自动创建，没有特殊需要可以不加
    ip_address = models.CharField(max_length=64)
    request_count = models.IntegerField(default=1)
    last_request_time = models.DateTimeField(auto_now_add=True)

    def __str__(self):
        return f"{self.id} - {self.ip_address} - {self.last_request_time}"

------
views.py

from django.shortcuts import render
from django.views.decorators.http import require_http_methods
from django.http import JsonResponse, HttpResponse
from pracdjango import models
from pracdjango.decorator import rate_limit

# Create your views here.
@rate_limit(requests_per_day=4)
@require_http_methods(["POST"])
def requests_ip_addr(request):

    return JsonResponse({'msg':'ok'},status = 200)

------
decorator.py

from django.http import HttpResponse
from django.utils import timezone
from pracdjango.models import IPRequestLog



def rate_limit(requests_per_day=3):
    
    def my_decorate(view_func):
        def wrapper(request,*args,**kwargs):
            ip_addr = request.META.get('REMOTE_ADDR')
            today = timezone.now().date()

            log , created = IPRequestLog.objects.get_or_create(
                ip_address = ip_addr,
                last_request_time__date = today,
                defaults = {'request_count':1,'last_request_time':timezone.now()}
            )
 
            # 判断是否已经有条目，如果已经有了则更新count 和 last_request_time
            if not created:
                if log.request_count >= requests_per_day:
                    return HttpResponse('Too Many Requests',status = 429)
                log.request_count +=1
            log.last_request_time = timezone.now()
            log.save()

            return view_func(request, *args, **kwargs)
        return wrapper
    return my_decorate


----response--
(when request more than 4 times)

Too Many Requests
--------------

总结：
1. IPRequestLog(models.Model) models 建表的时候已经要继承 models.Model
2. id 没有特殊需求可以不加，Django有机制会自动建立
3. 一开始想的是，用count的方式，计算表里同ip有多少个请求。但是写的过程中，如果限制数设定的很大，表里会有很多相似数据，不如直接在表里count。而且根据要求 4 看起来也是在表里直接统计request_count.而且这样也符合提示3 - 装饰器的应用。可以加在前面练习的views上
4. last_request_time 是DateTime 而 today是timezone ， datetime是带时分秒的，timezone只有日期，所以需要借用django的 __date 来自动lookup一下日期。